# 15 — Power Analysis: Minimum Detectable Effects for the Primary Spec

The headline result of this study is a null (0/35 RW survivors in the primary spec). A null
is only as informative as the design's power: "no detectable effect" bounds the true effect
only above the smallest effect the design could actually have detected. This notebook
quantifies that bound — the **minimum detectable effect (MDE)**: the smallest |β| (per 1 SD
of governance score, since every CG regressor is Van der Waerden-transformed within FY) that
would have been flagged with 80% probability.

Two detection standards, matching how the study actually reports:

1. **Raw** — a single test at p < .05 (two-sided), the naive standard.
2. **Romano-Wolf** — surviving the 35-test familywise correction, the study's actual
   inferential standard. The binding threshold is the bootstrap max-|t| critical value of
   the whole family, so the MDE is substantially larger.

Method: the textbook analytic MDE is `(z_{α/2} + z_{0.80}) × SE = 2.80 × SE`. A
cluster-bootstrap simulation check shows that formula is optimistic here (~70% empirical
power, not 80% — finite-sample noise in the clustered SE), so the headline numbers below are
**simulation-calibrated**: firms are resampled with replacement (the same cluster bootstrap
as the RW machinery), a known effect β is injected, and β is scaled until the empirical
rejection rate hits 80%. The analytic values are reported alongside for reference.

Inputs are the committed primary-spec outputs (`panel_regression_ready_primary.csv`,
`panel_individual_regressions_primary.csv`) — fully offline, no market-data access.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from scipy.stats import norm

BASE = Path.cwd()
PROC = BASE / 'data' / 'processed'

CG_CATS_PRIMARY = ['AINDEX', 'BINDEX', 'CINDEX', 'OINDEX', 'TRINDEX']
CTRL = ['Beta_Market', 'Momentum', 'Log_MarketCap', 'DE_Ratio']
OUTCOMES = ['mar_pct', 'capm_alpha', 'ff5_alpha', 'total_vol', 'idio_vol', 'downside_vol', 'roe']

Z_ALPHA = norm.ppf(0.975)          # 1.960 — raw two-sided 5%
Z_POWER = norm.ppf(0.80)           # 0.842 — target 80% power
RW_B    = 2000                     # bootstrap draws for the familywise critical value
N_SIMS  = 500                      # cluster-bootstrap sims per (pair, grid point)
GRID_M  = np.array([1.5, 2.3, 2.8, 3.4, 4.1, 5.0])   # candidate multipliers of SE
SEED    = 42

def winsorize(s, lo=0.01, hi=0.99):
    q_lo, q_hi = s.quantile([lo, hi])
    return s.clip(q_lo, q_hi)

def cluster_ols_fast(y, X, cl_codes, n_clusters, target_idx=1):
    n, k = X.shape
    XtX = X.T @ X
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta
    Xr = X * resid[:, None]
    S = np.zeros((n_clusters, k))
    np.add.at(S, cl_codes, Xr)
    meat = S.T @ S
    dof = (n_clusters / max(n_clusters - 1, 1)) * ((n - 1) / max(n - k, 1))
    vcov = XtX_inv @ meat @ XtX_inv * dof
    se = np.sqrt(np.maximum(np.diag(vcov), 0))
    t = beta[target_idx] / se[target_idx] if se[target_idx] > 0 else np.nan
    return beta[target_idx], t

def build_hyp_data(df, fe_cols, cats, controls=CTRL):
    hyp = {}
    for cat in cats:
        for y_col in OUTCOMES:
            cols = [y_col, cat] + controls + fe_cols + ['BSE Code']
            sub = df[cols].dropna().reset_index(drop=True)
            if len(sub) < 30:
                continue
            y = winsorize(sub[y_col]).values.astype(float)
            X = np.column_stack([np.ones(len(sub)), sub[cat].values,
                                 sub[controls].values, sub[fe_cols].values]).astype(float)
            firm_rows = pd.Series(np.arange(len(sub))).groupby(sub['BSE Code'].values).apply(np.array).to_dict()
            hyp[(cat, y_col)] = {'y': y, 'X': X, 'firm_rows': firm_rows,
                                 'firms': np.array(list(firm_rows.keys())), 'N': len(sub)}
    return hyp

def _row_cluster_codes(d):
    row_to_firm = np.empty(len(d['y']), dtype=object)
    for f, rows in d['firm_rows'].items():
        row_to_firm[rows] = f
    codes, uniq = pd.factorize(row_to_firm)
    return codes, len(uniq)

panel = pd.read_csv(PROC / 'panel_regression_ready_primary.csv')
fe = pd.concat([pd.get_dummies(panel['Industry'], prefix='_FE_IND', drop_first=True),
                pd.get_dummies(panel['FY'], prefix='_FE_FY', drop_first=True)], axis=1)
model_df = pd.concat([panel, fe], axis=1)
fe_cols = list(fe.columns)
hyp = build_hyp_data(model_df, fe_cols, CG_CATS_PRIMARY)

reg = pd.read_csv(PROC / 'panel_individual_regressions_primary.csv')
# cross-check: the fast estimator must reproduce the committed betas exactly
max_dev = max(abs(cluster_ols_fast(hyp[(r.Category, r.Outcome)]['y'], hyp[(r.Category, r.Outcome)]['X'],
                                   *_row_cluster_codes(hyp[(r.Category, r.Outcome)]))[0] - r.beta)
              for r in reg.itertuples())
print(f'Panel: {len(panel)} firm-FY rows, {panel["BSE Code"].nunique()} firms | '
      f'{len(hyp)} hypotheses | max |beta - committed| = {max_dev:.2e}')
assert max_dev < 1e-8, 'fast estimator does not reproduce committed betas'

Panel: 420 firm-FY rows, 210 firms | 35 hypotheses | max |beta - committed| = 2.95e-14


## 1 — Analytic MDE (reference)

`MDE = (1.96 + 0.842) × SE_cluster = 2.80 × SE`, per (sub-index, outcome), in the outcome's
native units per 1 SD of governance score. Optimistic — see §3 for the calibrated version.

In [2]:
reg['mde_analytic_raw'] = (Z_ALPHA + Z_POWER) * reg['se']
print('Analytic MDE — 80% power at raw p<.05 (per 1 SD of governance score):')
print(reg.pivot_table(index='Category', columns='Outcome', values='mde_analytic_raw')[OUTCOMES]
         .round(4).to_string())

Analytic MDE — 80% power at raw p<.05 (per 1 SD of governance score):
Outcome   mar_pct  capm_alpha  ff5_alpha  total_vol  idio_vol  downside_vol     roe
Category                                                                           
AINDEX     0.0556      0.0510     0.0777     0.0007    0.0007        0.0004  0.0216
BINDEX     0.0709      0.0643     0.0904     0.0010    0.0009        0.0005  0.0222
CINDEX     0.0679      0.0641     0.0712     0.0008    0.0007        0.0004  0.0151
OINDEX     0.0720      0.0666     0.0741     0.0008    0.0008        0.0005  0.0215
TRINDEX    0.0618      0.0591     0.0685     0.0009    0.0008        0.0005  0.0189


## 2 — Romano-Wolf familywise critical |t|

To survive the study's actual standard, an effect must clear not 1.96 but the bootstrap
max-|t| of the whole 35-test family (the stepdown's binding first stage — exact for the
top-ranked hypothesis, slightly conservative for lower ranks). Estimated with the same
firm-cluster bootstrap and B as the RW procedure in `09_regression.ipynb`.

In [3]:
keys = list(hyp.keys())
t_orig = np.array([cluster_ols_fast(hyp[k]['y'], hyp[k]['X'], *_row_cluster_codes(hyp[k]))[1] for k in keys])
rng = np.random.default_rng(SEED)
all_firms = np.unique(np.concatenate([hyp[k]['firms'] for k in keys]))
t0 = time.time()
max_abs_t = np.full(RW_B, np.nan)
for b in range(RW_B):
    sampled = rng.choice(all_firms, size=len(all_firms), replace=True)
    ts = np.full(len(keys), np.nan)
    for si, k in enumerate(keys):
        d = hyp[k]
        present = [f for f in sampled if f in d['firm_rows']]
        if not present:
            continue
        idx = np.concatenate([d['firm_rows'][f] for f in present])
        firm_seq = np.concatenate([[i] * len(d['firm_rows'][f]) for i, f in enumerate(present)])
        _, t_b = cluster_ols_fast(d['y'][idx], d['X'][idx], firm_seq, len(present))
        ts[si] = t_b
    max_abs_t[b] = np.nanmax(np.abs(ts - t_orig))
C_RW = float(np.nanquantile(max_abs_t, 0.90))
print(f'RW familywise critical |t| (90th pctile of bootstrap max-|t|, B={RW_B}, {time.time()-t0:.0f}s): '
      f'c_RW = {C_RW:.3f}  (vs raw 1.96)')
print(f'Analytic MDE inflation, RW vs raw: {(C_RW + Z_POWER) / (Z_ALPHA + Z_POWER):.2f}x')

RW familywise critical |t| (90th pctile of bootstrap max-|t|, B=2000, 76s): c_RW = 3.213  (vs raw 1.96)
Analytic MDE inflation, RW vs raw: 1.45x


## 3 — Simulation-calibrated MDE

For each (sub-index, outcome): strip the (near-zero) estimated effect from the winsorized
outcome, resample firms with replacement (common random numbers — the same `N_SIMS` resample
draws are reused across all grid points, so the power curve is smooth), inject `β = m × SE`,
re-estimate with clustered SEs, and record rejection at both thresholds (|t| ≥ 1.96 and
|t| ≥ c_RW). The 80% crossing of each power curve (probit-scale interpolation) gives the
calibrated multiplier, and `MDE = m* × SE`.

In [4]:
def calibrated_mde_row(d, se_hat, rng):
    beta_full, _ = cluster_ols_fast(d['y'], d['X'], *_row_cluster_codes(d))
    y0 = d['y'] - beta_full * d['X'][:, 1]
    firms = d['firms']
    sims = []
    for s in range(N_SIMS):  # pre-draw resamples once (CRN across the grid)
        sampled = rng.choice(firms, size=len(firms), replace=True)
        idx = np.concatenate([d['firm_rows'][f] for f in sampled])
        firm_seq = np.concatenate([[i] * len(d['firm_rows'][f]) for i, f in enumerate(sampled)])
        sims.append((idx, firm_seq, len(sampled)))
    pow_raw, pow_rw = [], []
    for m in GRID_M:
        beta_inj = m * se_hat
        ts = np.full(N_SIMS, np.nan)
        for s, (idx, firm_seq, n_cl) in enumerate(sims):
            y_sim = y0[idx] + beta_inj * d['X'][idx, 1]
            _, t_s = cluster_ols_fast(y_sim, d['X'][idx], firm_seq, n_cl)
            ts[s] = t_s
        ok = np.isfinite(ts)
        pow_raw.append(np.mean(np.abs(ts[ok]) >= Z_ALPHA))
        pow_rw.append(np.mean(np.abs(ts[ok]) >= C_RW))
    def crossing(powers):
        z = norm.ppf(np.clip(powers, 0.01, 0.99))
        return float(np.interp(norm.ppf(0.80), z, GRID_M))
    return crossing(pow_raw), crossing(pow_rw)

t0 = time.time()
rows = []
rng = np.random.default_rng(SEED + 1)
for cat in CG_CATS_PRIMARY:
    for y_col in OUTCOMES:
        if (cat, y_col) not in hyp:
            continue
        r = reg[(reg['Category'] == cat) & (reg['Outcome'] == y_col)].iloc[0]
        m_raw, m_rw = calibrated_mde_row(hyp[(cat, y_col)], r['se'], rng)
        rows.append({'Category': cat, 'Outcome': y_col, 'N': int(r['N']), 'se': r['se'],
                     'beta_hat': r['beta'],
                     'mde_analytic_raw': (Z_ALPHA + Z_POWER) * r['se'],
                     'm_star_raw': m_raw, 'mde_sim_raw': m_raw * r['se'],
                     'm_star_rw': m_rw, 'mde_sim_rw': m_rw * r['se']})
    print(f'  {cat} done ({time.time()-t0:.0f}s)')
mde = pd.DataFrame(rows)
print(f'\nCalibrated {len(mde)} pairs in {time.time()-t0:.0f}s '
      f'({N_SIMS} sims x {len(GRID_M)} grid points each)')
print(f"Calibrated multiplier range: raw {mde['m_star_raw'].min():.2f}-{mde['m_star_raw'].max():.2f} "
      f"(analytic 2.80), RW {mde['m_star_rw'].min():.2f}-{mde['m_star_rw'].max():.2f}")

  AINDEX done (21s)


  BINDEX done (43s)


  CINDEX done (70s)


  OINDEX done (94s)


  TRINDEX done (115s)

Calibrated 35 pairs in 115s (500 sims x 6 grid points each)
Calibrated multiplier range: raw 2.69-3.61 (analytic 2.80), RW 3.96-5.00


In [5]:
print('Simulation-calibrated MDE — 80% power at raw p<.05:')
print(mde.pivot_table(index='Category', columns='Outcome', values='mde_sim_raw')[OUTCOMES].round(4).to_string())
print('\nSimulation-calibrated MDE — 80% power to clear the RW familywise threshold:')
print(mde.pivot_table(index='Category', columns='Outcome', values='mde_sim_rw')[OUTCOMES].round(4).to_string())

mde['c_rw'] = C_RW
mde.to_csv(PROC / 'power_mde_primary.csv', index=False)
print('\nSaved -> data/processed/power_mde_primary.csv')

Simulation-calibrated MDE — 80% power at raw p<.05:
Outcome   mar_pct  capm_alpha  ff5_alpha  total_vol  idio_vol  downside_vol     roe
Category                                                                           
AINDEX     0.0615      0.0597     0.1001     0.0008    0.0008        0.0005  0.0225
BINDEX     0.0779      0.0705     0.1072     0.0010    0.0010        0.0006  0.0234
CINDEX     0.0749      0.0746     0.0740     0.0007    0.0007        0.0004  0.0145
OINDEX     0.0854      0.0842     0.0945     0.0010    0.0009        0.0005  0.0224
TRINDEX    0.0769      0.0685     0.0879     0.0010    0.0009        0.0006  0.0222

Simulation-calibrated MDE — 80% power to clear the RW familywise threshold:
Outcome   mar_pct  capm_alpha  ff5_alpha  total_vol  idio_vol  downside_vol     roe
Category                                                                           
AINDEX     0.0895      0.0842     0.1386     0.0011    0.0011        0.0007  0.0335
BINDEX     0.1123      0.1017  

## 4 — What the design could and could not see

Converting the RW-standard MDEs into the outcomes' natural units (the summary below is
computed live from the table above). The reading: **moderate-to-large effects were
detectable; economically small ones were not.** The null therefore rules out sub-index →
outcome effects above roughly the printed thresholds, but says nothing about smaller ones —
and the survivorship conditioning documented in the README's Limitations §1 attenuates any
true effect toward zero before it even reaches this detection floor.

In [6]:
med = mde.groupby('Outcome')[['mde_sim_raw', 'mde_sim_rw']].median().reindex(OUTCOMES)
print('Median MDE across the 5 sub-indices, per 1 SD of governance score:')
print('=' * 76)
UNITS = {'mar_pct': ('annual market-adjusted return', 100, 'pp'),
         'capm_alpha': ('annual CAPM alpha', 100, 'pp'),
         'ff5_alpha': ('annual FF5+MOM alpha', 100, 'pp'),
         'total_vol': ('daily total volatility', 1e4, 'bp'),
         'idio_vol': ('daily idiosyncratic volatility', 1e4, 'bp'),
         'downside_vol': ('daily downside volatility', 1e4, 'bp'),
         'roe': ('ROE', 100, 'pp')}
for y in OUTCOMES:
    label, scale, unit = UNITS[y]
    print(f'  {y:13s} {label:32s} raw: {med.loc[y, "mde_sim_raw"]*scale:7.2f} {unit}   '
          f'RW: {med.loc[y, "mde_sim_rw"]*scale:7.2f} {unit}')
print('=' * 76)
print(f'(raw = single test at p<.05; RW = survives the {len(mde)}-test family at c_RW = {C_RW:.2f})')

Median MDE across the 5 sub-indices, per 1 SD of governance score:
  mar_pct       annual market-adjusted return    raw:    7.69 pp   RW:   11.02 pp
  capm_alpha    annual CAPM alpha                raw:    7.05 pp   RW:   10.17 pp
  ff5_alpha     annual FF5+MOM alpha             raw:    9.45 pp   RW:   13.22 pp
  total_vol     daily total volatility           raw:    9.53 bp   RW:   13.90 bp
  idio_vol      daily idiosyncratic volatility   raw:    9.12 bp   RW:   13.17 bp
  downside_vol  daily downside volatility        raw:    5.31 bp   RW:    7.67 bp
  roe           ROE                              raw:    2.24 pp   RW:    3.27 pp
(raw = single test at p<.05; RW = survives the 35-test family at c_RW = 3.21)
